In [1]:

from collections import deque

# Appendix A: Company network
network = {
    "N01": ["N02"],
    "N02": ["N01", "N03", "N04", "N30"],
    "N03": ["N02", "N04", "N05", "N06", "N07", "N12", "N13"],
    "N04": ["N02", "N03", "N08", "N09", "N10", "N11", "N14", "N15"],
    "N05": ["N03"],
    "N06": ["N03"],
    "N07": ["N03"],
    "N08": ["N04", "N09", "N18", "N23"],
    "N09": ["N04", "N08"],
    "N10": ["N04", "N11", "N17"],
    "N11": ["N04", "N10"],
    "N12": ["N03", "N16", "N17"],
    "N13": ["N03", "N18", "N19", "N20"],
    "N14": ["N04", "N21", "N22", "N23"],
    "N15": ["N04", "N24", "N25", "N26", "N27"],
    "N16": ["N12"],
    "N17": ["N10", "N12"],
    "N18": ["N08", "N13"],
    "N19": ["N13"],
    "N20": ["N13"],
    "N21": ["N14"],
    "N22": ["N14"],
    "N23": ["N08", "N14"],
    "N24": ["N15"],
    "N25": ["N15"],
    "N26": ["N15"],
    "N27": ["N15", "N28", "N29"],
    "N28": ["N27"],
    "N29": ["N27"],
    "N30": ["N02"]
}

device_names = {
    "N01": "Internet-Gateway",
    "N02": "Firewall-Main",
    "N03": "Core-Switch-1",
    "N04": "Core-Switch-2",
    "N05": "Web-Server",
    "N06": "Mail-Server",
    "N07": "DNS-Server",
    "N08": "Database-Server",
    "N09": "Auth-Server",
    "N10": "File-Server",
    "N11": "Backup-Server",
    "N12": "Switch-HR",
    "N13": "Switch-Finance",
    "N14": "Switch-Dev",
    "N15": "Switch-Lab",
    "N16": "HR-PC-1",
    "N17": "HR-PC-2",
    "N18": "Finance-PC-1",
    "N19": "Finance-PC-2",
    "N20": "Finance-Printer",
    "N21": "Dev-PC-1",
    "N22": "Dev-PC-2",
    "N23": "Build-Server",
    "N24": "Lab-PC-1",
    "N25": "Lab-PC-2",
    "N26": "Lab-PC-3",
    "N27": "WiFi-AP",
    "N28": "Guest-Laptop-1",
    "N29": "Guest-Laptop-2",
    "N30": "IDS-Sensor"
}

def bfs(graph, start, ignored=None):
    visited = {start}
    queue = deque([start])
    parent = {start: None}
    distance = {start: 0}
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbour in graph[node]:
            if neighbour == ignored:
                continue
            if neighbour not in visited:
                visited.add(neighbour)
                parent[neighbour] = node
                distance[neighbour] = distance[node] + 1
                queue.append(neighbour)

    return order, parent, distance

def build_path(parent, goal):
    if goal not in parent:
        return None

    path = []
    while goal is not None:
        path.append(goal)
        goal = parent[goal]

    return path[::-1]

def show_names(ids):
    return [device_names[n] for n in ids]

# Part A: Infection spread from N28
order, parent, distance = bfs(network, "N28")

print("1. BFS infection order:")
print(order)

print("\n2. Infection levels:")
for hour in range(max(distance.values()) + 1):
    devices = [n for n in order if distance[n] == hour]
    print("Hop", hour, ":", devices,
          "| Count:", len(devices))

print("\n3. Devices infected within each time limit:")
for hours in [2, 3, 4]:
    infected = [n for n in order if distance[n] <= hours]
    print(hours, "hours:", infected)
    print("Device names:", show_names(infected))

print("\n4. Database-Server infection time:",
      distance["N08"], "hours")

last_hour = max(distance.values())
last_devices = [
    n for n in order if distance[n] == last_hour
]
print("Farthest devices:", last_devices)
print("Names:", show_names(last_devices))
print("Last device in BFS order:", order[-1])

# Part B: Shortest paths
for goal in ["N08", "N16"]:
    path = build_path(parent, goal)
    print("\nBFS path to", device_names[goal])
    print(path)
    print(show_names(path))
    print("Hops:", len(path) - 1)

# Part C: BFS from the Internet-Gateway
gateway_order, _, gateway_distance = bfs(network, "N01")
farthest = max(gateway_distance.values())
print("\n7. Farthest from Internet-Gateway:",
      farthest, "hops")
print([
    n for n in gateway_order
    if gateway_distance[n] == farthest
])

# Part C: Remove one device at a time
for removed in ["N04", "N03", "N02"]:
    reached, _, _ = bfs(network, "N01", ignored=removed)
    unreachable = [
        n for n in network if n not in reached
    ]
    print("\nRemoving", removed,
          device_names[removed])
    print("Unreachable IDs:", unreachable)
    print("Unreachable names:", show_names(unreachable))
    print("Number unreachable:", len(unreachable))


1. BFS infection order:
['N28', 'N27', 'N15', 'N29', 'N04', 'N24', 'N25', 'N26', 'N02', 'N03', 'N08', 'N09', 'N10', 'N11', 'N14', 'N01', 'N30', 'N05', 'N06', 'N07', 'N12', 'N13', 'N18', 'N23', 'N17', 'N21', 'N22', 'N16', 'N19', 'N20']

2. Infection levels:
Hop 0 : ['N28'] | Count: 1
Hop 1 : ['N27'] | Count: 1
Hop 2 : ['N15', 'N29'] | Count: 2
Hop 3 : ['N04', 'N24', 'N25', 'N26'] | Count: 4
Hop 4 : ['N02', 'N03', 'N08', 'N09', 'N10', 'N11', 'N14'] | Count: 7
Hop 5 : ['N01', 'N30', 'N05', 'N06', 'N07', 'N12', 'N13', 'N18', 'N23', 'N17', 'N21', 'N22'] | Count: 12
Hop 6 : ['N16', 'N19', 'N20'] | Count: 3

3. Devices infected within each time limit:
2 hours: ['N28', 'N27', 'N15', 'N29']
Device names: ['Guest-Laptop-1', 'WiFi-AP', 'Switch-Lab', 'Guest-Laptop-2']
3 hours: ['N28', 'N27', 'N15', 'N29', 'N04', 'N24', 'N25', 'N26']
Device names: ['Guest-Laptop-1', 'WiFi-AP', 'Switch-Lab', 'Guest-Laptop-2', 'Core-Switch-2', 'Lab-PC-1', 'Lab-PC-2', 'Lab-PC-3']
4 hours: ['N28', 'N27', 'N15', 'N29', 